# Gladius · local tutorial

Make native cutouts, a median display stack and a GIF from two synthetic images,
then explore the GUI inside this notebook. No MAST token or observations are
needed. The synthetic source stays at the same sky position while the simulated
pointing shifts it on the detector.

Install Gladius with the `notebook` extra in this kernel's environment. These
cells require Gladius 0.1.0a3 or later.
The source checkout instructions in the docs install the current implementation.


## 1 · Choose a work directory

The demo writes input ASDF images and outputs beneath this directory. On later
runs it reuses the existing demo configuration; it never overwrites your source
images. Choose another directory for a separate example.


In [ ]:
from pathlib import Path

WORKSPACE = Path.cwd() / "gladius-local-example"


In [ ]:
import json

from gladius import load_config, run
from gladius.demo import create_demo

config_path = WORKSPACE / "demo.yaml"
if not config_path.exists():
    config_path = create_demo(WORKSPACE)

# RunConfig contains targets, input selection, processing choices and output paths.
config = load_config(config_path)
manifest_path = run(config, progress=print)

# run() returns a Path. The JSON at that path records status and cutout files.
manifest = json.loads(manifest_path.read_text())
print("Run status:", manifest["status"])
print("Run directory:", manifest_path.parent)


## 2 · Inspect the images

These are plain image previews. The display stretch affects their appearance,
not the stored science pixels. The median stack combines sky-aligned exposures;
the GIF cycles through individual aligned exposures. Both are for inspection,
not scientific measurements.


In [ ]:
import numpy as np
from astropy.io import fits
from astropy.visualization import AsinhStretch
from IPython.display import display, Image as DisplayImage
from PIL import Image
from gladius.io import read_cutout

run_directory = manifest_path.parent
entry = next(iter(manifest["entries"].values()))
cutout_path = run_directory / next(p for p in entry["files"] if p.endswith(".asdf"))
cutout = read_cutout(cutout_path)
science = cutout.arrays["data"]
low, high = np.nanpercentile(science, [1, 99.95])

def preview(array):
    normalized = np.clip((array - low) / max(high - low, 1e-12), 0, 1)
    stretched = AsinhStretch(0.03)(np.nan_to_num(normalized))
    return Image.fromarray((255 * stretched[::-1]).astype("uint8")).resize(
        (256, 256), Image.Resampling.NEAREST
    )

print("Individual cutout")
display(preview(science))
displays = json.loads((run_directory / "stack-manifest.json").read_text())
with fits.open(run_directory / displays["products"][0]["path"]) as hdus:
    print("Median display stack")
    display(preview(hdus[0].data))
print("Individual exposures, animated")
display(DisplayImage(filename=str(run_directory / displays["animations"][0]["path"])))


## 3 · Explore the GUI

Set `NOTEBOOK_ORIGIN` below to **just the scheme, host and port** in your notebook's
browser address, for example `http://localhost:8888` or `http://127.0.0.1:8889`.
Do not include `/lab/...`, query parameters or a login token. An exact origin is
required because the notebook and GUI use different local ports.

The next cell starts the GUI with the demo settings and target already loaded,
without starting another run.
The form's help and defaults explain the YAML. You can download a modified YAML
or click **Make cutouts** to run it. For real data, supply your own inputs and
credentials as described in the documentation.


In [ ]:
import threading

from IPython.display import IFrame, display
from gladius.web import GuiServer

NOTEBOOK_ORIGIN = "http://localhost:8888"  # Match your notebook browser address.

if "gui_thread" not in globals() or not gui_thread.is_alive():
    gui_server = GuiServer(WORKSPACE, port=0, iframe_origin=NOTEBOOK_ORIGIN, initial_config=config)
    gui_thread = threading.Thread(target=gui_server.serve_forever, daemon=True)
    gui_thread.start()

gladius_ui_url = gui_server.url + "/"
print("Open the GUI in a separate tab:", gladius_ui_url)
display(IFrame(src=gladius_ui_url, width="100%", height=650))


## 4 · Stop the embedded server

Closing the frame does not stop the GUI or its run. Wait for **Complete** or
**Failed** before running this cell. Restarting the kernel also stops this
embedded server. Your image files and output records remain on disk.

If the frame is blocked, verify `NOTEBOOK_ORIGIN`, stop the server with the cell
below, then rerun the GUI cell with the corrected origin. You can always use the
printed standalone link instead. This local notebook expects Jupyter to be on
the same computer as your browser; use the Nexus tutorial for Nexus.


In [ ]:
if "gui_thread" in globals() and gui_thread.is_alive():
    gui_server.shutdown()
    gui_server.server_close()
    gui_thread.join(timeout=5)
    print("Embedded GUI stopped.")
